In [1]:
!pip install -q pytorch-lightning datasets transformers evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 37.8 MB/s eta 0:00:00


In [2]:
# Instalacion de librerias
import random
import torch
import numpy as np
import os
from pytorch_lightning import seed_everything
import matplotlib.pyplot as plt
import seaborn as sns
import re

seed_val = 42
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)# Store the average loss after eachepoch so we can plot them.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["TF_DETERMINISTIC_OPS"] = "1" # See:https://github.com/NVIDIA/tensorflow-determinism#confirmed-current-gpu-specific-sources-of-non-determinism-with-solutions
seed_everything(42, workers=True)

from datasets import Dataset, DatasetDict #, load_metric EN PRINCIPIO ESTÁ DESCONTINUADO, TENDREMOS QUE BUSCAR OTRA ALTERNATIVA
import pandas as pd
import sklearn as sk
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, f1_score
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, \
TrainingArguments, Trainer, pipeline, EarlyStoppingCallback
from huggingface_hub import login

INFO:lightning_fabric.utilities.seed:Seed set to 42


In [3]:
# Comprobación de disponibilidad de GPU
if torch.cuda.is_available():
    # Si hay una GPU disponible, la asignamos como dispositivo principal
    device = torch.device("cuda")
    print(f'✅ GPU detectada. Trabajando con: "{torch.cuda.get_device_name(0)}"')
else:
    # Si no hay GPU, el modelo y los tensores irán a la CPU
    device = torch.device("cpu")
    print('⚠️ Trabajando con CPU.')
    print('Para usar la GPU en Colab: Ve al menú superior "Entorno de ejecución" -> "Cambiar tipo de entorno de ejecución" -> Selecciona "GPU T4".')

# Opcional pero recomendado: Ver cuánta memoria VRAM tienes disponible
if device.type == 'cuda':
    memoria_total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Memoria VRAM total disponible: {memoria_total:.2f} GB')

⚠️ Trabajando con CPU.
Para usar la GPU en Colab: Ve al menú superior "Entorno de ejecución" -> "Cambiar tipo de entorno de ejecución" -> Selecciona "GPU T4".


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Lectura y etiquetado de Datos

In [5]:
# ================================
# 1. Carga y preparación de datos
# ================================
# Cargar dataset de entrenamiento con etiquetas
#df = pd.read_csv("/home/adrian/Escritorio/DeepSexist/DatasetManagement/EXIST2025DatasetV0.3/EXIST 2025 Videos Dataset/training/EXIST2025_training_task3_1_DISCARD.csv")
df = pd.read_csv("/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_training_3_1.csv")
df = df.rename(columns={"label_task_3_1_merged": "label"})
df = df.drop(columns=["Unnamed: 0"])
df["label"] = df["label"].astype(int)

# from sklearn.model_selection import train_test_split
# train_df, eval_df = train_test_split(df, test_size=0.15, stratify=df["label"], random_state=42)

# train_dataset = Dataset.from_pandas(train_df)
# eval_dataset = Dataset.from_pandas(eval_df)

# PENDIENTE DE DEFINIR UN FICHERO TRAIN, VAL Y TEST
# División estratificada
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

print("Distribución fichero de entrenamiento:")
print(train_df.value_counts('label'))

print("Distribución fichero de test:")
print(test_df.value_counts('label'))

print("Distribución fichero de validación:")
print(val_df.value_counts('label'))


Distribución fichero de entrenamiento:
label
0    914
1    841
Name: count, dtype: int64
Distribución fichero de test:
label
0    196
1    181
Name: count, dtype: int64
Distribución fichero de validación:
label
0    196
1    180
Name: count, dtype: int64


# Preprocesado del texto

In [7]:
# Para ver el vocabulario del dataset

import pandas as pd

# Suponiendo que la columna de texto se llama 'text'
# Tokeniza y obtiene todas las palabras
vocabulario = set(
    palabra.lower()
    for fila in df["text"].dropna()
    for palabra in fila.split()
)

# Mostrar todo el vocabulario
print(vocabulario)

# (Opcional) Ver el número de palabras únicas
print(f"\nTotal de palabras únicas: {len(vocabulario)}")


{'gun,', '~jeshuruucberhen', 'preguntó', 'day."""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""', 'manager', 'conociste', 'dijiste,', 'ggierdr', 'tenerla', 'manejando', '0906.', 'blonde,', 'qqu', 'true!', 'train', 'encontrarás', 'justificante', 'niño!', 'noche?hoy', 'carried', 'preocupan.el', 'to.', 'noeonnl?', 'traidor,', 'houston,', 'isj', 'independiente', '"sticogro', 'frowning_face_with_open_mouth', 'erkläre', 'consolarlos', '€89', 'pibiñas.ay', 'leagues', 'marico!', 'kitchen?', 'jarns', 'denuncie', 'jim2o', 'creo,', 'patriarcal.ire', 'mes,', 'señalar', 'uli,', 'hubiese', 'profes', '1960s', 'sirve', 'usarlo', 'out.', 'battle.', 'aap', 'bobby,', 'vean', 'jornada', 'shiny', '¡suscríbete!', '

In [8]:
import re

def remove_links(tweet):
    """Takes a string and removes web links from it"""
    tweet = re.sub(r'http\S+', '', tweet)        # remove http links
    tweet = re.sub(r'bit.ly/\S+', '', tweet)     # remove bitly links
    tweet = re.sub(r'\[link\]', '', tweet )      # remove [link]
    tweet = re.sub(r'\[url\]', '', tweet )       # remove [url]
    tweet = re.sub(r'pic.twitter\S+','', tweet)
    return tweet

def remove_users(tweet):
    """Takes a string and removes retweet and @user information"""
    tweet = re.sub('(RT\s@[A-Za-z]+[A-Za-z0-9-_]+)', '', tweet)  # remove re-tweet
    tweet = re.sub('(@[A-Za-z]+[A-Za-z0-9-_]+)', '', tweet)      # remove tweeted at
    tweet = re.sub(r'\[user\]', '', tweet )                      # remove [user]
    return tweet

def remove_hashtags(tweet):
    """Takes a string and removes any hash tags"""
    tweet = re.sub('(#[A-Za-z]+[A-Za-z0-9-_]+)', '', tweet)      # remove hash tags
    return tweet

def remove_av(tweet):
    """Takes a string and removes AUDIO/VIDEO tags or labels"""
    tweet = re.sub('VIDEO:', '', tweet)  # remove 'VIDEO:' from start of tweet
    tweet = re.sub('AUDIO:', '', tweet)  # remove 'AUDIO:' from start of tweet
    return tweet

def remove_emojis(tweet):
    emoj = re.compile("["
        u"\U00002700-\U000027BF"  # Dingbats
        u"\U0001F600-\U0001F64F"  # Emoticons
        u"\U00002600-\U000026FF"  # Miscellaneous Symbols
        u"\U0001F300-\U0001F5FF"  # Miscellaneous Symbols And Pictographs
        u"\U0001F900-\U0001F9FF"  # Supplemental Symbols and Pictographs
        u"\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
        u"\U00010000-\U0010FFFF"
        u"\U0001F680-\U0001F6FF"  # Transport and Map Symbols
        u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        u"\U0001f926-\U0001f937"
        u"\U00010000-\U0010ffff"
        u"\u2640-\u2642"
        u"\u2600-\u2B55"
        u"\ufe0f"  # dingbats

                      "]+", re.UNICODE)
    return re.sub(emoj, '', tweet)

<>:14: SyntaxWarning: invalid escape sequence '\s'
<>:14: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_690/1710960392.py:14: SyntaxWarning: invalid escape sequence '\s'
  tweet = re.sub('(RT\s@[A-Za-z]+[A-Za-z0-9-_]+)', '', tweet)  # remove re-tweet
